## Tag 03 - MLP - Fortgeschrittene

In [1]:
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 3: Mehrschichtige Netze (MLP)
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Implement a full MLP with backpropagation from scratch
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

class MLP:
    """MLP mit Vorwärts- und Rückwärtsdurchlauf (NumPy)"""
    
    def __init__(self, schichten, lernrate=0.01):
        self.lernrate = lernrate
        self.gewichte = []
        self.biase = []
        for i in range(len(schichten)-1):
            W = np.random.randn(schichten[i], schichten[i+1]) * np.sqrt(2.0/schichten[i])
            b = np.zeros((1, schichten[i+1]))
            self.gewichte.append(W); self.biase.append(b)
    
    def relu(self, x): return np.maximum(0, x)
    def relu_abl(self, x): return (x > 0).astype(float)
    def sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def vorwaerts(self, X):
        """Vorwärtsdurchlauf durch alle Schichten"""
        self.aktivierungen = [X]
        self.pre_aktivierungen = []
        a = X
        for i, (W, b) in enumerate(zip(self.gewichte, self.biase)):
            z = a @ W + b
            self.pre_aktivierungen.append(z)
            a = self.relu(z) if i < len(self.gewichte)-1 else self.sigmoid(z)
            self.aktivierungen.append(a)
        return a
    
    def rueckwaerts(self, y):
        """Rückwärtsdurchlauf: Gradienten berechnen und Gewichte aktualisieren"""
        m = y.shape[0]
        delta = self.aktivierungen[-1] - y.reshape(-1, 1)
        for i in range(len(self.gewichte)-1, -1, -1):
            dW = self.aktivierungen[i].T @ delta / m
            db = delta.mean(axis=0, keepdims=True)
            self.gewichte[i] -= self.lernrate * dW
            self.biase[i] -= self.lernrate * db
            if i > 0:
                delta = (delta @ self.gewichte[i].T) * self.relu_abl(self.pre_aktivierungen[i-1])
    
    def trainieren(self, X, y, epochen=200):
        verluste = []
        for _ in range(epochen):
            ausgabe = self.vorwaerts(X)
            ausgabe = np.clip(ausgabe, 1e-15, 1-1e-15)
            verlust = -np.mean(y * np.log(ausgabe.flatten()) + (1-y) * np.log(1-ausgabe.flatten()))
            verluste.append(verlust)
            self.rueckwaerts(y)
        return verluste

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

mlp = MLP([2, 16, 8, 1], lernrate=0.1)
verluste = mlp.trainieren(X_train, y_train, epochen=500)

y_pred = (mlp.vorwaerts(X_test).flatten() > 0.5).astype(int)
genauigkeit = (y_pred == y_test).mean()
print(f"Testgenauigkeit (NumPy MLP): {genauigkeit:.2%}")

plt.figure(figsize=(8,5))
plt.plot(verluste, 'b-', linewidth=2)
plt.title('MLP Lernkurve (NumPy Implementation)'); plt.xlabel('Epoche'); plt.ylabel('BCE Verlust')
plt.grid(True, alpha=0.3)
plt.show()


Testgenauigkeit (NumPy MLP): 97.00%
Lernkurve gespeichert: mlp_numpy_lernkurve.png


C:\Users\esmae\AppData\Local\Temp\ipykernel_3552\904667654.py:89: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [2]:
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 3: Mehrschichtige Netze (MLP)
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Grid search for optimal MLP hyperparameters
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import itertools

tf.random.set_seed(42); np.random.seed(42)

X, y = make_moons(n_samples=600, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

def erstelle_mlp(n_schichten, n_neuronen, lernrate):
    """MLP mit variabler Tiefe, Breite und Lernrate"""
    schichten = [tf.keras.layers.Dense(n_neuronen, activation='relu', input_shape=(2,))]
    for _ in range(n_schichten - 1):
        schichten.append(tf.keras.layers.Dense(n_neuronen, activation='relu'))
    schichten.append(tf.keras.layers.Dense(1, activation='sigmoid'))
    m = tf.keras.Sequential(schichten)
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lernrate), loss='binary_crossentropy', metrics=['accuracy'])
    return m

ergebnisse = []
# Grid Search über Hyperparameter
for n_s, n_n, lr in itertools.product([1, 2, 3], [8, 16, 32], [0.01, 0.001]):
    modell = erstelle_mlp(n_s, n_n, lr)
    modell.fit(X_train, y_train, epochs=50, verbose=0)
    _, acc = modell.evaluate(X_test, y_test, verbose=0)
    ergebnisse.append({'schichten': n_s, 'neuronen': n_n, 'lr': lr, 'genauigkeit': acc})
    print(f"Schichten={n_s}, Neuronen={n_n}, LR={lr:.3f} → Genauigkeit={acc:.2%}")

bestes = max(ergebnisse, key=lambda x: x['genauigkeit'])
print(f"\nBeste Konfiguration: {bestes}")

# Heatmap der Ergebnisse (LR=0.01)
acc_matrix = np.zeros((3, 3))
konf_labels_x = [8, 16, 32]; konf_labels_y = [1, 2, 3]
for r in ergebnisse:
    if r['lr'] == 0.01:
        i = konf_labels_y.index(r['schichten'])
        j = konf_labels_x.index(r['neuronen'])
        acc_matrix[i,j] = r['genauigkeit']

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(acc_matrix, cmap='YlOrRd', vmin=0.8, vmax=1.0)
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels([8,16,32]); ax.set_yticklabels([1,2,3])
ax.set_xlabel('Neuronen pro Schicht'); ax.set_ylabel('Anzahl Schichten')
ax.set_title('Grid Search Heatmap (LR=0.01)')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{acc_matrix[i,j]:.2%}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


Schichten=1, Neuronen=8, LR=0.010 → Genauigkeit=95.00%


Schichten=1, Neuronen=8, LR=0.001 → Genauigkeit=84.17%


Schichten=1, Neuronen=16, LR=0.010 → Genauigkeit=95.00%


Schichten=1, Neuronen=16, LR=0.001 → Genauigkeit=86.67%


Schichten=1, Neuronen=32, LR=0.010 → Genauigkeit=95.00%


...

Schichten=3, Neuronen=16, LR=0.001 → Genauigkeit=96.67%


Schichten=3, Neuronen=32, LR=0.010 → Genauigkeit=95.83%


Schichten=3, Neuronen=32, LR=0.001 → Genauigkeit=97.50%

Beste Konfiguration: {'schichten': 3, 'neuronen': 32, 'lr': 0.001, 'genauigkeit': 0.9750000238418579}


Grid Search Heatmap gespeichert: hyperparameter_grid.png


C:\Users\esmae\AppData\Local\Temp\ipykernel_3552\2972336692.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 3: Mehrschichtige Netze (MLP)
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Analyze feature importance in MLP using permutation method
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42); np.random.seed(42)

# Breast Cancer Dataset
daten = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(daten.data, daten.target, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

# MLP Modell
modell = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
modell.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modell.fit(X_train, y_train, epochs=100, verbose=0)

basis_acc = modell.evaluate(X_test, y_test, verbose=0)[1]
print(f"Basisgenauigkeit: {basis_acc:.2%}")

# Permutation Importance berechnen: Feature permutieren → Genauigkeitsabfall messen
wichtigkeiten = []
for feat_idx in range(X_test.shape[1]):
    X_permutiert = X_test.copy()
    np.random.shuffle(X_permutiert[:, feat_idx])  # Feature zufällig permutieren
    perm_acc = modell.evaluate(X_permutiert, y_test, verbose=0)[1]
    wichtigkeiten.append(basis_acc - perm_acc)

wichtigkeiten = np.array(wichtigkeiten)
sortiert = np.argsort(wichtigkeiten)[::-1]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(wichtigkeiten)), wichtigkeiten[sortiert],
       color=['red' if w > 0 else 'blue' for w in wichtigkeiten[sortiert]], alpha=0.7)
ax.set_xticks(range(len(wichtigkeiten)))
ax.set_xticklabels([daten.feature_names[i][:15] for i in sortiert], rotation=45, ha='right', fontsize=8)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_ylabel('Genauigkeitsabfall (Permutation Importance)')
ax.set_title('MLP Feature Importance – Breast Cancer Dataset')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Basisgenauigkeit: 96.49%


Feature Importance gespeichert: feature_importance.png


C:\Users\esmae\AppData\Local\Temp\ipykernel_3552\1011445210.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
